# 👥 HR Attrition Analytics — Complete Notebook




## Install Libraries


In [ ]:
!pip install pandas numpy plotly scipy -q
print("✅ All packages installed successfully")

✅ All packages installed successfully


## Import Libraries



In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

import plotly.io as pio
pio.renderers.default = "colab"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ All libraries imported")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

✅ All libraries imported
   pandas  : 2.2.2
   numpy   : 2.0.2



## Load & Combine


In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# Tag source BEFORE combining — gives us traceability
train["_source"] = "train"
test["_source"]  = "test"

df_raw = pd.concat([train, test], ignore_index=True)

print("=" * 45)
print(f"  Train    : {train.shape[0]:>7,} rows × {train.shape[1]} cols")
print(f"  Test     : {test.shape[0]:>7,} rows × {test.shape[1]} cols")
print(f"  Combined : {df_raw.shape[0]:>7,} rows × {df_raw.shape[1]} cols")
print("=" * 45)

print("\n📋 First 3 rows (raw):")
display(df_raw.head(3))

print("\n📋 Raw column names:")
print(df_raw.columns.tolist())

  Train    :  59,598 rows × 25 cols
  Test     :  14,900 rows × 25 cols
  Combined :  74,498 rows × 25 cols

📋 First 3 rows (raw):


,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Overtime,Distance from Home,Education Level,Marital Status,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition,_source
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,No,22,Associate Degree,Married,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed,train
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,No,21,Master’s Degree,Divorced,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed,train
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,No,11,Bachelor’s Degree,Married,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed,train



📋 Raw column names:
['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition', '_source']


## Normalize Column Names



In [ ]:
def to_snake(col: str) -> str:
    """Convert column name to snake_case."""
    return (col.strip()
               .lower()
               .replace(" ", "_")
               .replace("-", "_")
               .replace("/", "_")
               .replace("'", ""))

df = df_raw.copy()
original_cols = df.columns.tolist()
df.columns    = [to_snake(c) for c in df.columns]

print("Column name mapping:")
print(f"  {'BEFORE':<35} → {'AFTER'}")
print("  " + "-" * 55)
for orig, new in zip(original_cols, df.columns):
    changed = " ← changed" if orig != new else ""
    print(f"  {orig:<35} → {new}{changed}")

Column name mapping:
  BEFORE                              → AFTER
  -------------------------------------------------------
  Employee ID                         → employee_id ← changed
  Age                                 → age ← changed
  Gender                              → gender ← changed
  Years at Company                    → years_at_company ← changed
  Job Role                            → job_role ← changed
  Monthly Income                      → monthly_income ← changed
  Work-Life Balance                   → work_life_balance ← changed
  Job Satisfaction                    → job_satisfaction ← changed
  Performance Rating                  → performance_rating ← changed
  Number of Promotions                → number_of_promotions ← changed
  Overtime                            → overtime ← changed
  Distance from Home                  → distance_from_home ← changed
  Education Level                     → education_level ← changed
  Marital Status                      → ma

## Full Data Audit



In [ ]:
print("=" * 60)
print(f"  SHAPE: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── Data types ──────────────────────────────────────────
print("\n📌 DATA TYPES:")
dtype_df = df.dtypes.reset_index()
dtype_df.columns = ["column", "dtype"]
dtype_df["category"] = dtype_df["dtype"].apply(
    lambda x: "🔢 numeric" if x in ["int64","float64"] else "🔤 text/categorical"
)
display(dtype_df)

# ── Null counts ─────────────────────────────────────────
print("\n📌 NULL COUNTS:")
nulls = df.isnull().sum()
null_df = nulls[nulls > 0].reset_index()
null_df.columns = ["column", "null_count"]
null_df["pct_missing"] = (null_df["null_count"] / len(df) * 100).round(2)
if len(null_df):
    display(null_df)
else:
    print("  ✅ No nulls found in raw data")

# ── Duplicates ───────────────────────────────────────────
n_dup = df.duplicated().sum()
n_dup_id = df.duplicated(subset=["employee_id"]).sum() if "employee_id" in df.columns else "N/A"
print(f"\n📌 DUPLICATES:")
print(f"  Full row duplicates : {n_dup:,}")
print(f"  Employee ID dupes   : {n_dup_id:,}  ← cross-file employees counted twice")

# ── Target distribution ─────────────────────────────────
print(f"\n📌 TARGET — Attrition (raw values):")
att_counts = df["attrition"].value_counts(dropna=False)
att_pct    = df["attrition"].value_counts(normalize=True, dropna=False) * 100
att_summary = pd.DataFrame({"count": att_counts, "pct": att_pct.round(1)})
display(att_summary)

# ── Categorical value audit ──────────────────────────────
print("\n📌 CATEGORICAL VALUE AUDIT — all unique values per column:")
cat_cols = ["work_life_balance", "job_satisfaction", "performance_rating",
            "education_level", "job_level", "company_size", "company_reputation",
            "employee_recognition", "gender", "marital_status", "job_role",
            "remote_work", "leadership_opportunities", "innovation_opportunities"]

audit_rows = []
for col in cat_cols:
    if col in df.columns:
        unique_vals = sorted(df[col].dropna().unique().tolist())
        audit_rows.append({
            "column"      : col,
            "n_unique"    : df[col].nunique(),
            "null_count"  : df[col].isnull().sum(),
            "unique_values": str(unique_vals),
        })

display(pd.DataFrame(audit_rows))

# ── Numeric summary ─────────────────────────────────────
print("\n📌 NUMERIC SUMMARY:")
num_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[num_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2))

  SHAPE: 74,498 rows × 25 columns

📌 DATA TYPES:


,column,dtype,category
0,employee_id,int64,🔢 numeric
1,age,int64,🔢 numeric
2,gender,object,🔤 text/categorical
3,years_at_company,int64,🔢 numeric
4,job_role,object,🔤 text/categorical
5,monthly_income,int64,🔢 numeric
6,work_life_balance,object,🔤 text/categorical
7,job_satisfaction,object,🔤 text/categorical
8,performance_rating,object,🔤 text/categorical
9,number_of_promotions,int64,🔢 numeric



📌 NULL COUNTS:
  ✅ No nulls found in raw data

📌 DUPLICATES:
  Full row duplicates : 0
  Employee ID dupes   : 0  ← cross-file employees counted twice

📌 TARGET — Attrition (raw values):


,count,pct
attrition,,
Stayed,39128,52.50
Left,35370,47.50



📌 CATEGORICAL VALUE AUDIT — all unique values per column:


,column,n_unique,null_count,unique_values
0,work_life_balance,4,0,"['Excellent', 'Fair', 'Good', 'Poor']"
1,job_satisfaction,4,0,"['High', 'Low', 'Medium', 'Very High']"
2,performance_rating,4,0,"['Average', 'Below Average', 'High', 'Low']"
3,education_level,5,0,"['Associate Degree', 'Bachelor’s Degree', 'Hig..."
4,job_level,3,0,"['Entry', 'Mid', 'Senior']"
5,company_size,3,0,"['Large', 'Medium', 'Small']"
6,company_reputation,4,0,"['Excellent', 'Fair', 'Good', 'Poor']"
7,employee_recognition,4,0,"['High', 'Low', 'Medium', 'Very High']"
8,gender,2,0,"['Female', 'Male']"
9,marital_status,3,0,"['Divorced', 'Married', 'Single']"



📌 NUMERIC SUMMARY:


,count,mean,std,min,1%,25%,50%,75%,99%,max
employee_id,74498.00,37249.50,21505.86,1.00,745.97,18625.25,37249.50,55873.75,73753.03,74498.00
age,74498.00,38.53,12.08,18.00,18.00,28.00,39.00,49.00,59.00,59.00
years_at_company,74498.00,15.72,11.22,1.00,1.00,7.00,13.00,23.00,45.00,51.00
monthly_income,74498.00,7299.38,2152.51,1226.00,3138.00,5652.00,7348.00,8876.00,12147.21,16149.00
number_of_promotions,74498.00,0.83,1.00,0.00,0.00,0.00,1.00,2.00,4.00,4.00
distance_from_home,74498.00,49.99,28.51,1.00,2.00,25.00,50.00,75.00,98.00,99.00
number_of_dependents,74498.00,1.65,1.55,0.00,0.00,0.00,1.00,3.00,6.00,6.00
company_tenure,74498.00,55.73,25.40,2.00,7.00,36.00,56.00,76.00,110.00,128.00


## Fix Target Column: Attrition

Converts the `attrition` column from text strings (`"Stayed"` / `"Left"`) to integers (`0` / `1`).



In [ ]:
print("📋 Raw attrition unique values:")
raw_vals = df["attrition"].value_counts(dropna=False)
display(raw_vals.reset_index().rename(columns={"count":"rows"}))

# Map strings to integers
df["attrition"] = df["attrition"].map({"Stayed": 0, "Left": 1})

# Validate — catch any values that didn't map
unmapped_n = df["attrition"].isnull().sum()
if unmapped_n > 0:
    print(f"\n⚠️  WARNING: {unmapped_n:,} rows had unrecognized attrition values → removed")
    df = df.dropna(subset=["attrition"])

df["attrition"] = df["attrition"].astype(int)

# Summary
total  = len(df)
left   = int(df["attrition"].sum())
stayed = total - left
rate   = left / total * 100

print(f"\n✅ Attrition successfully encoded as 0/1 integer")
print(f"   Stayed (0) : {stayed:>8,}  ({100-rate:.1f}%)")
print(f"   Left   (1) : {left:>8,}  ({rate:.1f}%)")
print(f"   ─────────────────────────────")
print(f"   Total      : {total:>8,}")

📋 Raw attrition unique values:


,attrition,rows
0,Stayed,39128
1,Left,35370



✅ Attrition successfully encoded as 0/1 integer
   Stayed (0) :   39,128  (52.5%)
   Left   (1) :   35,370  (47.5%)
   ─────────────────────────────
   Total      :   74,498


## Normalize Unicode Text




In [ ]:
def normalize_text(series: pd.Series) -> pd.Series:
    """
    Normalize unicode characters in a string Series to plain ASCII equivalents.

    The Kaggle HR dataset uses curly/smart quotes (U+2018, U+2019) in values
    like "Bachelor's Degree". These look identical to straight apostrophes on
    screen but are different characters in Python string comparison.

    This function must be applied BEFORE any .map() encoding or .mode() calls.
    """
    return (series.astype(str)
                  .str.replace('\u2019', "'", regex=False)  # ' RIGHT SINGLE QUOTATION MARK
                  .str.replace('\u2018', "'", regex=False)  # ' LEFT SINGLE QUOTATION MARK
                  .str.replace('\u201c', '"', regex=False)  # " LEFT DOUBLE QUOTATION MARK
                  .str.replace('\u201d', '"', regex=False)  # " RIGHT DOUBLE QUOTATION MARK
                  .str.replace('\u2013', '-', regex=False)  # – EN DASH
                  .str.replace('\u2014', '-', regex=False)  # — EM DASH
                  .str.strip())                              # Remove leading/trailing whitespace

# Apply to all text (object dtype) columns
string_cols = df.select_dtypes(include="object").columns.tolist()
for col in string_cols:
    df[col] = normalize_text(df[col])

print(f"✅ Normalized unicode in {len(string_cols)} text columns")
print(f"   Columns processed: {string_cols}")

# Verify education_level now has plain apostrophes
print(f"\n📋 education_level values after normalization:")
display(df["education_level"].value_counts().reset_index())

# Check repr of first value — should show straight apostrophe '
sample = df["education_level"].dropna().iloc[0]
print(f"\n   Sample value repr: {repr(sample)}")
has_curly = '\u2019' in sample or '\u2018' in sample
print(f"   Curly apostrophes remaining: {'YES ⚠️' if has_curly else 'NO ✅'}")

✅ Normalized unicode in 16 text columns
   Columns processed: ['gender', 'job_role', 'work_life_balance', 'job_satisfaction', 'performance_rating', 'overtime', 'education_level', 'marital_status', 'job_level', 'company_size', 'remote_work', 'leadership_opportunities', 'innovation_opportunities', 'company_reputation', 'employee_recognition', '_source']

📋 education_level values after normalization:


,education_level,count
0,Bachelor's Degree,22331
1,Associate Degree,18649
2,Master's Degree,15021
3,High School,14680
4,PhD,3817



   Sample value repr: 'Associate Degree'
   Curly apostrophes remaining: NO ✅


## Ordinal Encoding

### What this cell does
Creates numeric companion columns (`_num` suffix) for all ordinal categorical features.

### Why create numeric versions?
- **Charts need numbers** for correlation calculations
- **Pearson correlation** requires numeric input — can't correlate `"Good"` with attrition
- We **keep the original string columns** for chart labels (so bars say "Good", not "4")
- The `_num` columns are only used for statistical calculations

### The corrected 5-level scales
The Kaggle data dictionary was incomplete. The real dataset has:

| Column | Dict said | Reality |
|--------|-----------|---------|
| `work_life_balance` | 4 levels | **5 levels** — includes `"Fair"` |
| `job_satisfaction` | 4 levels | **5 levels** — includes `"Very High"` |
| `company_reputation` | 4 levels | **5 levels** — includes `"Fair"` |
| `employee_recognition` | 4 levels | **5 levels** — includes `"Very High"` |

### The validation block
After every `.map()`, we check if any values produced `NaN`. If they did, it means the data has a category our map doesn't cover — and we print exactly which values are unmapped so you can add them. This is how we discovered the 5th levels in v1.

In [ ]:
# Education Level — full names (with both apostrophe variants, though Cell 9 normalizes them)
edu_map = {
    "High School"        : 1,
    "Associate"          : 2, "Associate's Degree"  : 2, "Associate Degree"  : 2,
    "Bachelor's"         : 3, "Bachelor's Degree"   : 3, "Bachelor Degree"   : 3,
    "Master's"           : 4, "Master's Degree"     : 4, "Masters Degree"    : 4,
    "PhD"                : 5, "Doctorate"           : 5,
}

# CORRECTED MAPS — include all 5 levels discovered from real data
ordinal_maps = {
    # 5 levels (Fair added between Below Average=2 and Good=4)
    "work_life_balance"   : {"Poor":1,"Below Average":2,"Fair":3,"Good":4,"Excellent":5},

    # 5 levels (Very High added above High=4)
    "job_satisfaction"    : {"Very Low":1,"Low":2,"Medium":3,"High":4,"Very High":5},

    # 4 levels — matches data dictionary exactly
    "performance_rating"  : {"Low":1,"Below Average":2,"Average":3,"High":4},

    # Covers all degree name variants (unicode fixed in Cell 9)
    "education_level"     : edu_map,

    # 3 levels — matches data dictionary exactly
    "job_level"           : {"Entry":1,"Mid":2,"Senior":3},
    "company_size"        : {"Small":1,"Medium":2,"Large":3},

    # 5 levels (Fair added between Poor=2 and Good=4)
    "company_reputation"  : {"Very Poor":1,"Poor":2,"Fair":3,"Good":4,"Excellent":5},

    # 5 levels (Very High added above High=4)
    "employee_recognition": {"Very Low":1,"Low":2,"Medium":3,"High":4,"Very High":5},
}

# Apply maps with validation
encoding_report = []
all_clean = True

for col, mapping in ordinal_maps.items():
    if col not in df.columns:
        encoding_report.append({"column":col,"levels":0,"status":"⚠️  column not found","unmapped":0})
        continue

    df[f"{col}_num"] = df[col].map(mapping)
    n_unmapped = int(df[f"{col}_num"].isnull().sum())

    if n_unmapped > 0:
        bad_vals = df[col][df[f"{col}_num"].isnull()].unique().tolist()
        print(f"❌ {col}: {n_unmapped:,} unmapped → {bad_vals}")
        # Fallback to median of successfully mapped values
        fallback = df[f"{col}_num"].median()
        df[f"{col}_num"] = df[f"{col}_num"].fillna(fallback)
        encoding_report.append({"column":col,"levels":df[col].nunique(),
                                 "status":"❌ fallback used","unmapped":n_unmapped})
        all_clean = False
    else:
        max_level = int(df[f"{col}_num"].max())
        encoding_report.append({"column":col,"levels":df[col].nunique(),
                                 "status":"✅ clean","unmapped":0})

# Binary Yes/No → 0/1 with safety net
for col in ["remote_work","leadership_opportunities","innovation_opportunities"]:
    if col in df.columns:
        df[col] = df[col].map({"Yes":1,"No":0}).fillna(0).astype(int)

print("\n📋 Encoding Report:")
display(pd.DataFrame(encoding_report))

if all_clean:
    print("\n🎉 All categories mapped cleanly — zero unmapped values!")
else:
    print("\n⚠️  Some unmapped values found — update the maps above and re-run")


📋 Encoding Report:


,column,levels,status,unmapped
0,work_life_balance,4,✅ clean,0
1,job_satisfaction,4,✅ clean,0
2,performance_rating,4,✅ clean,0
3,education_level,5,✅ clean,0
4,job_level,3,✅ clean,0
5,company_size,3,✅ clean,0
6,company_reputation,4,✅ clean,0
7,employee_recognition,4,✅ clean,0



🎉 All categories mapped cleanly — zero unmapped values!


## Outlier Capping (Winsorization)

### What this cell does
Clips extreme values in continuous numeric columns to the 1st–99th percentile range.

### Why cap outliers?
Synthetic datasets often have extreme outliers because the random generation process isn't constrained by real-world logic. These extremes:
- Distort chart scales (a single $1M income makes all other bars look flat)
- Inflate standard deviations, making distributions look wider than they are
- Can pull correlation values toward or away from zero artificially

### Why 1st–99th percentile, not IQR fencing?
- IQR fencing (1.5×IQR) is very aggressive and can cap 5–10% of data
- 1st–99th percentile capping (Winsorization) removes only the most extreme 1% on each end
- We keep 98% of the data untouched — much more conservative

### Why NOT cap age or promotions?
- `age` has a hard natural range (18–60) defined by the data dictionary
- `number_of_promotions` is a count that should stay as-is
- `years_at_company` is bounded by age — no need to cap



In [ ]:
cap_report = []

for col in ["monthly_income","distance_from_home","company_tenure"]:
    if col not in df.columns:
        continue

    p01   = df[col].quantile(0.01)
    p99   = df[col].quantile(0.99)
    orig_min = df[col].min()
    orig_max = df[col].max()
    n_low  = int((df[col] < p01).sum())
    n_high = int((df[col] > p99).sum())

    df[col] = df[col].clip(p01, p99)

    cap_report.append({
        "column"         : col,
        "original_range" : f"{orig_min:,.0f} – {orig_max:,.0f}",
        "capped_range"   : f"{df[col].min():,.0f} – {df[col].max():,.0f}",
        "rows_capped_low": n_low,
        "rows_capped_high":n_high,
        "total_capped"   : n_low + n_high,
    })

print("📋 Outlier Capping Summary:")
display(pd.DataFrame(cap_report))
print("\n✅ Outlier capping complete — all rows preserved, only extreme values adjusted")

📋 Outlier Capping Summary:


,column,original_range,capped_range,rows_capped_low,rows_capped_high,total_capped
0,monthly_income,"1,226 – 16,149","3,138 – 12,147",742,745,1487
1,distance_from_home,1 – 99,2 – 98,737,710,1447
2,company_tenure,2 – 128,7 – 110,592,729,1321



✅ Outlier capping complete — all rows preserved, only extreme values adjusted


## Feature Engineering




- `monthly_income` tells you a salary number
- `income_percentile_in_role` tells you if that person is underpaid RELATIVE to their peers — much more meaningful for understanding attrition

### Features created

| Feature | Formula | Why it's useful |
|---------|---------|----------------|
| `age_group` | `pd.cut(age, bins)` → ordered Categorical | Groups for demographic charts. **Must be ordered Categorical** or charts sort alphabetically (26-35 before 18-25). |
| `income_annual` | `monthly_income × 12` | HR thinks in annual salaries, not monthly |
| `tenure_gap` | `company_tenure − years_at_company` | Years of industry experience BEFORE joining. High value = experienced hire. Low value = grew up here. |
| `income_pct_in_role` | percentile rank of income within same job_role | Detects underpaid employees relative to peers in the same role — a key retention risk |
| `is_early_career` | `age ≤ 30 AND job_level == "Entry"` | Flags the highest-mobility segment — young workers in entry roles have most options |
| `low_engagement` | `wlb_num ≤ 2 AND js_num ≤ 2` | Composite at-risk flag: poor balance AND dissatisfied simultaneously |

In [ ]:
# ── Age groups — MUST be ordered Categorical ──────────────────
# Without ordered=True, charts will sort: "18-25", "26-35", "36-45", "46-60"
# alphabetically — correct here but wrong for "Poor","Fair","Good" style labels
age_labels = ["18-25","26-35","36-45","46-60"]
df["age_group"] = pd.Categorical(
    pd.cut(df["age"], bins=[17,25,35,45,60], labels=age_labels),
    categories=age_labels, ordered=True,
)

# ── Annual income ─────────────────────────────────────────────
df["income_annual"] = (df["monthly_income"] * 12).round(0)

# ── Tenure gap: industry exp before this company ──────────────
# Clip at 0 because company_tenure < years_at_company is a data inconsistency
df["tenure_gap"] = (df["company_tenure"] - df["years_at_company"]).clip(lower=0)

# ── Income percentile within job role ─────────────────────────
# A $5,000/mo income means very different things for Entry vs Senior Finance
df["income_pct_in_role"] = (
    df.groupby("job_role")["monthly_income"]
      .transform(lambda x: x.rank(pct=True).mul(100).round(1))
)

# ── Early career flag ─────────────────────────────────────────
df["is_early_career"] = (
    (df["age"] <= 30) & (df["job_level"] == "Entry")
).astype(int)

# ── Low engagement composite flag ─────────────────────────────
if "work_life_balance_num" in df.columns and "job_satisfaction_num" in df.columns:
    df["low_engagement"] = (
        (df["work_life_balance_num"] <= 2) & (df["job_satisfaction_num"] <= 2)
    ).astype(int)

# ── Summary ───────────────────────────────────────────────────
new_cols = ["age_group","income_annual","tenure_gap","income_pct_in_role",
            "is_early_career","low_engagement"]
print("✅ Feature engineering complete")
print(f"   Added {len(new_cols)} new features\n")

for col in new_cols:
    if col in df.columns:
        sample = df[col].dropna().value_counts().head(3).to_dict() if df[col].dtype == "object" or str(df[col].dtype) == "category" else None
        print(f"   {col:<25} dtype={str(df[col].dtype):<15} nunique={df[col].nunique()}")

print(f"\n   Early career employees  : {df['is_early_career'].sum():,} ({df['is_early_career'].mean()*100:.1f}%)")
if "low_engagement" in df.columns:
    print(f"   Low-engagement employees: {df['low_engagement'].sum():,} ({df['low_engagement'].mean()*100:.1f}%)")
print(f"\n   Final shape: {df.shape}")

✅ Feature engineering complete
   Added 6 new features

   age_group                 dtype=category        nunique=4
   income_annual             dtype=float64         nunique=8782
   tenure_gap                dtype=int64           nunique=79
   income_pct_in_role        dtype=float64         nunique=1001
   is_early_career           dtype=int64           nunique=2
   low_engagement            dtype=int64           nunique=2

   Early career employees  : 9,144 (12.3%)
   Low-engagement employees: 994 (1.3%)

   Final shape: (74498, 39)


## Final Quality Checklist



In [ ]:
# Drop _source — we no longer need it
if "_source" in df.columns:
    df = df.drop(columns=["_source"])

print("=" * 60)
print("  FINAL DATA QUALITY CHECKLIST  (8 checks)")
print("=" * 60)

checks = []

checks.append((
    "_source column dropped",
    "_source" not in df.columns,
    "still present" if "_source" in df.columns else ""
))

total_nulls = df.isnull().sum().sum()
checks.append((
    "Zero nulls in any column",
    total_nulls == 0,
    f"{total_nulls:,} nulls remain" if total_nulls else ""
))

att_set = set(df["attrition"].unique())
checks.append((
    "Attrition is exactly {0, 1}",
    att_set == {0,1},
    f"Found: {att_set}"
))

num_expected = [f"{c}_num" for c in ordinal_maps if c in df.columns]
missing_num  = [c for c in num_expected if c not in df.columns]
nulls_num    = sum(df[c].isnull().sum() for c in num_expected if c in df.columns)
checks.append((
    "All _num columns present & filled",
    len(missing_num) == 0 and nulls_num == 0,
    f"Missing: {missing_num} | nulls in _num: {nulls_num}"
))

neg_inc = (df["monthly_income"] < 0).sum()
checks.append((
    "No negative monthly_income",
    neg_inc == 0,
    f"{neg_inc} rows with negative income" if neg_inc else ""
))

bad_age = ((df["age"] < 18) | (df["age"] > 60)).sum()
checks.append((
    "All ages in valid range [18, 60]",
    bad_age == 0,
    f"{bad_age} ages out of range" if bad_age else ""
))

dup_ids = df.duplicated(subset=["employee_id"]).sum()
checks.append((
    "No duplicate employee_id",
    dup_ids == 0,
    f"{dup_ids} duplicate IDs remain" if dup_ids else ""
))

binary_ok = all(
    set(df[c].unique()).issubset({0,1})
    for c in ["remote_work","leadership_opportunities","innovation_opportunities"]
    if c in df.columns
)
checks.append((
    "Binary columns are 0/1 integers",
    binary_ok,
    "Some binary columns have unexpected values" if not binary_ok else ""
))

all_passed = True
for name, passed, detail in checks:
    icon = "✅" if passed else "❌"
    note = f"  ← {detail}" if detail and not passed else ""
    print(f"  {icon}  {name:<45}{note}")
    if not passed: all_passed = False

print()
if all_passed:
    print("🎉 All 8 checks passed — data is clean and ready for EDA!")
else:
    print("⚠️  Fix failed checks before proceeding to EDA")

print(f"\n📐 Final shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🎯 Attrition rate: {df['attrition'].mean()*100:.1f}%")

  FINAL DATA QUALITY CHECKLIST  (8 checks)
  ✅  _source column dropped                       
  ✅  Zero nulls in any column                     
  ✅  Attrition is exactly {0, 1}                  
  ✅  All _num columns present & filled            
  ✅  No negative monthly_income                   
  ✅  All ages in valid range [18, 60]             
  ✅  No duplicate employee_id                     
  ✅  Binary columns are 0/1 integers              

🎉 All 8 checks passed — data is clean and ready for EDA!

📐 Final shape : 74,498 rows × 38 columns
🎯 Attrition rate: 47.5%


---
# 🟩 EXPLORATORY DATA ANALYSIS
---

## Cell 15 — Helper Functions


### `attrition_rate()` — why rates, not counts?
**Wrong approach — using raw counts:**
- Healthcare: 800 left
- Technology: 200 left
→ Conclusion: "Healthcare has 4× worse attrition!" (WRONG)

**Correct approach — using rates:**
- Healthcare: 800 left out of 4,000 = 20% rate
- Technology: 200 left out of 500 = 40% rate
→ Conclusion: "Technology actually has 2× worse attrition!" (CORRECT)

Groups have different sizes. Rates are the only fair comparison.

### `chi_square_test()` — why do we need statistical tests?
A bar chart shows WHAT the attrition rates are. A chi-square test tells you WHETHER the differences between groups are statistically significant or just random chance. If a test returns `p < 0.05`, the pattern is real, not noise.

In [ ]:
def attrition_rate(data, col, order=None):
    """
    Compute attrition rate (%) per group.

    Returns DataFrame with: col, rate_pct, headcount, left_count
    Sorted by rate_pct descending by default, or by order if provided.

    Args:
        data  : DataFrame containing the column and attrition
        col   : Column name to group by
        order : List of category values in desired display order (for ordinal cols)
    """
    agg = (
        data.groupby(col, observed=True)["attrition"]
            .agg(["mean","count","sum"])
            .rename(columns={"mean":"rate_pct","count":"headcount","sum":"left_count"})
            .reset_index()
            .assign(rate_pct=lambda x: (x["rate_pct"] * 100).round(2))
    )
    if order:
        agg[col] = pd.Categorical(agg[col], categories=order, ordered=True)
        agg = agg.sort_values(col)
    else:
        agg = agg.sort_values("rate_pct", ascending=False)
    return agg


def chi_square_test(data, col):
    """
    Run Chi-square test of independence between a categorical column and attrition.

    Null hypothesis: the column and attrition are INDEPENDENT (no relationship).
    If p < 0.05: reject null → the relationship is statistically significant.

    Returns: chi2 statistic, p-value, degrees of freedom, significance string
    """
    contingency = pd.crosstab(data[col], data["attrition"])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    sig = "*** (p<0.001)" if p < 0.001 else "** (p<0.01)" if p < 0.01 else "* (p<0.05)" if p < 0.05 else "ns (not significant)"
    return chi2, p, dof, sig


print("✅ Helper functions defined: attrition_rate(), chi_square_test()")

✅ Helper functions defined: attrition_rate(), chi_square_test()


## Overall Attrition KPI

what fraction of all employees left the company.



Attrition Rate = (employees left/all employees) × 100


In [ ]:
total  = len(df)
left   = int(df["attrition"].sum())
stayed = total - left
rate   = left / total * 100

print("=" * 50)
print(f"  OVERALL ATTRITION RATE")
print("=" * 50)
print(f"  Total employees : {total:,}")
print(f"  Left (attrition): {left:,}  ({rate:.1f}%)")
print(f"  Stayed          : {stayed:,}  ({100-rate:.1f}%)")
print("=" * 50)
print(f"\n  👉 This {rate:.1f}% is the BASELINE for all comparisons below.")
print(f"     Any group above {rate:.1f}% is higher-than-average risk.")

fig = go.Figure(go.Pie(
    labels=["Stayed","Left"], values=[stayed,left], hole=0.55,
    marker_colors=["#22c55e","#ef4444"],
    textinfo="label+percent+value", textfont_size=14,
    hovertemplate="%{label}: %{value:,} employees (%{percent})<extra></extra>",
))
fig.update_layout(
    title_text="Overall Employee Attrition",
    annotations=[dict(text=f"<b>{rate:.0f}%</b><br>attrition<br>rate",
                      x=0.5, y=0.5, font_size=18, showarrow=False, font_color="#0d1b2a")],
    template="plotly_white", height=450,
    legend=dict(orientation="h", y=-0.05),
)
fig.show()

  OVERALL ATTRITION RATE
  Total employees : 74,498
  Left (attrition): 35,370  (47.5%)
  Stayed          : 39,128  (52.5%)

  👉 This 47.5% is the BASELINE for all comparisons below.
     Any group above 47.5% is higher-than-average risk.


## Attrition by Demographics

Attrition rates broken down by Gender, Marital Status, and Age Group.


In [ ]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=["By Gender","By Marital Status","By Age Group"])

demo_configs = [
    ("gender",        None,                              1),
    ("marital_status",None,                              2),
    ("age_group",     ["18-25","26-35","36-45","46-60"], 3),
]

for col, order, idx in demo_configs:
    agg = attrition_rate(df, col, order)
    # Add overall rate reference line
    fig.add_hline(y=rate, line_dash="dot", line_color="gray",
                  annotation_text=f"Overall: {rate:.1f}%", row=1, col=idx)
    fig.add_trace(go.Bar(
        x=agg[col].astype(str), y=agg["rate_pct"],
        text=agg["rate_pct"].astype(str) + "%", textposition="outside",
        marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg["rate_pct"]],
        customdata=agg[["headcount","left_count"]],
        hovertemplate=("%{x}<br>Rate: %{y:.1f}%<br>"
                       "Headcount: %{customdata[0]:,}<br>"
                       "Left: %{customdata[1]:,}<extra></extra>"),
        showlegend=False,
    ), row=1, col=idx)

fig.update_layout(title_text="Attrition Rate by Demographics (red = above overall rate)",
                  height=460, template="plotly_white")
fig.update_yaxes(title_text="Attrition Rate (%)", row=1, col=1)
fig.show()

# Statistical significance
print("\n📊 Chi-Square Significance Tests:")
for col, order, _ in demo_configs:
    chi2, p, dof, sig = chi_square_test(df, col)
    print(f"  {col:<20} χ²={chi2:>8.2f}  p={p:.4f}  {sig}")


📊 Chi-Square Significance Tests:
  gender               χ²=  754.10  p=0.0000  *** (p<0.001)
  marital_status       χ²= 6046.96  p=0.0000  *** (p<0.001)
  age_group            χ²=  252.90  p=0.0000  *** (p<0.001)


## Attrition by Job Factors



In [ ]:
wlb_order = ["Poor","Below Average","Fair","Good","Excellent"]
js_order  = ["Very Low","Low","Medium","High","Very High"]
pr_order  = ["Low","Below Average","Average","High"]

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Attrition by Job Role",
    "Attrition by Work-Life Balance (5 levels)",
    "Attrition by Job Satisfaction (5 levels)",
    "Attrition by Performance Rating",
])

configs = [
    ("job_role",          None,       1, 1),
    ("work_life_balance", wlb_order,  1, 2),
    ("job_satisfaction",  js_order,   2, 1),
    ("performance_rating",pr_order,   2, 2),
]

for col, order, row, col_idx in configs:
    agg = attrition_rate(df, col, order)
    fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=row, col=col_idx)
    fig.add_trace(go.Bar(
        x=agg[col].astype(str), y=agg["rate_pct"],
        text=agg["rate_pct"].astype(str) + "%", textposition="outside",
        marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg["rate_pct"]],
        showlegend=False,
    ), row=row, col=col_idx)

fig.update_layout(title_text="Attrition Rate by Job & Workplace Factors",
                  height=700, template="plotly_white",
                  margin=dict(t=60, b=40))
fig.show()

print("\n📊 Chi-Square Significance Tests:")
for col, order, _, __ in configs:
    chi2, p, dof, sig = chi_square_test(df, col)
    print(f"  {col:<25} χ²={chi2:>10.2f}  p={p:.5f}  {sig}")


📊 Chi-Square Significance Tests:
  job_role                  χ²=     14.78  p=0.00519  ** (p<0.01)
  work_life_balance         χ²=   2912.50  p=0.00000  *** (p<0.001)
  job_satisfaction          χ²=    388.24  p=0.00000  *** (p<0.001)
  performance_rating        χ²=    257.11  p=0.00000  *** (p<0.001)


##  Attrition by Seniority & Education



In [ ]:
edu_order = ["High School","Associate's Degree","Bachelor's Degree","Master's Degree","PhD"]
jl_order  = ["Entry","Mid","Senior"]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Attrition by Job Level","Attrition by Education Level"])

for col, order, idx in [("job_level",jl_order,1), ("education_level",edu_order,2)]:
    agg = attrition_rate(df, col, order)
    fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=1, col=idx)
    fig.add_trace(go.Bar(
        x=agg[col].astype(str), y=agg["rate_pct"],
        text=agg["rate_pct"].astype(str) + "%", textposition="outside",
        marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg["rate_pct"]],
        showlegend=False,
    ), row=1, col=idx)

fig.update_layout(title_text="Attrition by Seniority & Education",
                  height=450, template="plotly_white")
fig.update_yaxes(title_text="Attrition Rate (%)", row=1, col=1)
fig.show()

for col, order in [("job_level",jl_order),("education_level",edu_order)]:
    chi2, p, dof, sig = chi_square_test(df, col)
    print(f"  {col:<25} χ²={chi2:>10.2f}  p={p:.5f}  {sig}")

  job_level                 χ²=   7496.51  p=0.00000  *** (p<0.001)
  education_level           χ²=    859.32  p=0.00000  *** (p<0.001)


## Compensation & Tenure Analysis

### What this cell shows
How monthly income and years at the company differ between employees who stayed vs left.


- A violin plot shows the full distribution — you can see if income is bimodal (two salary clusters), skewed, or uniform
- The embedded box plot (white box inside the violin) still shows the median and IQR



In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Monthly Income Distribution","Years at Company"])

for idx, (yvar, ytitle) in enumerate([
    ("monthly_income",  "Monthly Income ($)"),
    ("years_at_company","Years at Company"),
], 1):
    for val, color, name in [(0,"#22c55e","Stayed"),(1,"#ef4444","Left")]:
        fig.add_trace(go.Violin(
            y=df[df.attrition==val][yvar], name=name,
            box_visible=True, meanline_visible=True,
            line_color=color, fillcolor=color, opacity=0.6,
            showlegend=(idx == 1),
        ), row=1, col=idx)

fig.update_layout(
    violinmode="group",
    title_text="Compensation & Tenure: Stayed vs Left",
    template="plotly_white", height=460,
)
fig.show()

# Median comparison table
print("\n📊 Median Comparison (Stayed vs Left):")
print(f"{'Column':<25} {'Stayed Median':>16} {'Left Median':>14} {'Difference':>12}")
print("-" * 70)
for col, label, fmt in [
    ("monthly_income",  "Monthly Income ($)", "${:>,.0f}"),
    ("income_annual",   "Annual Income ($)",  "${:>,.0f}"),
    ("years_at_company","Years at Company",   "{:>,.1f}"),
    ("age",             "Age",                "{:>,.1f}"),
]:
    if col in df.columns:
        s = df[df.attrition==0][col].median()
        l = df[df.attrition==1][col].median()
        diff = (l - s) / s * 100
        print(f"  {label:<23} {fmt.format(s):>16} {fmt.format(l):>14} {diff:>+11.1f}%")


📊 Median Comparison (Stayed vs Left):
Column                       Stayed Median    Left Median   Difference
----------------------------------------------------------------------
  Monthly Income ($)                $7,374         $7,319        -0.7%
  Annual Income ($)                $88,488        $87,828        -0.7%
  Years at Company                    14.0           12.0       -14.3%
  Age                                 39.0           38.0        -2.6%


## Promotions, Recognition & Remote Work



In [ ]:
rec_order = ["Very Low","Low","Medium","High","Very High"]

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["By Number of Promotions","By Employee Recognition","By Remote Work"])

# Promotions
agg_p = attrition_rate(df, "number_of_promotions")
fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=1, col=1)
fig.add_trace(go.Bar(
    x=agg_p["number_of_promotions"].astype(str), y=agg_p["rate_pct"],
    text=agg_p["rate_pct"].astype(str) + "%", textposition="outside",
    marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg_p["rate_pct"]],
    showlegend=False,
), row=1, col=1)

# Recognition
agg_r = attrition_rate(df, "employee_recognition", rec_order)
fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=1, col=2)
fig.add_trace(go.Bar(
    x=agg_r["employee_recognition"].astype(str), y=agg_r["rate_pct"],
    text=agg_r["rate_pct"].astype(str) + "%", textposition="outside",
    marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg_r["rate_pct"]],
    showlegend=False,
), row=1, col=2)

# Remote Work
df_rw = df.copy()
df_rw["remote_label"] = df_rw["remote_work"].map({1:"Remote",0:"On-site"})
agg_rw = attrition_rate(df_rw, "remote_label")
fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=1, col=3)
fig.add_trace(go.Bar(
    x=agg_rw["remote_label"].astype(str), y=agg_rw["rate_pct"],
    text=agg_rw["rate_pct"].astype(str) + "%", textposition="outside",
    marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg_rw["rate_pct"]],
    showlegend=False,
), row=1, col=3)

fig.update_layout(title_text="Promotions, Recognition & Remote Work vs Attrition",
                  height=460, template="plotly_white")
fig.update_yaxes(title_text="Attrition Rate (%)", row=1, col=1)
fig.show()

##  Distance from Home & Company Size


In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Distance from Home Distribution","Attrition by Company Size"])

# Distance histogram
for val, color, name in [(0,"#22c55e","Stayed"),(1,"#ef4444","Left")]:
    sub = df[df.attrition==val]["distance_from_home"]
    fig.add_trace(go.Histogram(
        x=sub, name=name, opacity=0.65,
        marker_color=color, nbinsx=30,
        hovertemplate=f"{name}<br>Distance: %{{x}}<br>Count: %{{y}}<extra></extra>",
    ), row=1, col=1)

# Company size
cs_order = ["Small","Medium","Large"]
agg_cs = attrition_rate(df, "company_size", cs_order)
fig.add_hline(y=rate, line_dash="dot", line_color="gray", row=1, col=2)
fig.add_trace(go.Bar(
    x=agg_cs["company_size"].astype(str), y=agg_cs["rate_pct"],
    text=agg_cs["rate_pct"].astype(str) + "%", textposition="outside",
    marker_color=["#ef4444" if r > rate else "#22c55e" for r in agg_cs["rate_pct"]],
    showlegend=False,
), row=1, col=2)

fig.update_layout(barmode="overlay", template="plotly_white", height=450,
                  title_text="Distance from Home & Company Size vs Attrition")
fig.update_xaxes(title_text="Distance (miles)", row=1, col=1)
fig.update_yaxes(title_text="Employee Count", row=1, col=1)
fig.update_yaxes(title_text="Attrition Rate (%)", row=1, col=2)
fig.show()

# Median distance comparison
s_dist = df[df.attrition==0]["distance_from_home"].median()
l_dist = df[df.attrition==1]["distance_from_home"].median()
print(f"\n📊 Median Distance from Home:")
print(f"   Stayed : {s_dist:.1f} miles")
print(f"   Left   : {l_dist:.1f} miles")
print(f"   Diff   : {l_dist-s_dist:+.1f} miles ({(l_dist-s_dist)/s_dist*100:+.1f}%)")


📊 Median Distance from Home:
   Stayed : 45.0 miles
   Left   : 55.0 miles
   Diff   : +10.0 miles (+22.2%)


## Heatmaps: Cross-Variable Patterns



In [ ]:
# ── Heatmap 1: Job Role × Job Satisfaction ────────────────
js_order  = ["Very Low","Low","Medium","High","Very High"]

pivot1 = (
    df.groupby(["job_role","job_satisfaction"], observed=True)["attrition"]
      .mean().mul(100).round(1).reset_index()
      .pivot(index="job_role", columns="job_satisfaction", values="attrition")
      .reindex(columns=[c for c in js_order if c in df["job_satisfaction"].unique()])
)

fig1 = px.imshow(pivot1, text_auto=".1f",
    color_continuous_scale=["#22c55e","#fcd34d","#ef4444"],
    aspect="auto",
    title="Heatmap 1: Attrition Rate (%) — Job Role × Job Satisfaction",
    labels=dict(color="Attrition %", x="Job Satisfaction", y="Job Role"),
)
fig1.update_layout(height=380, template="plotly_white")
fig1.show()

# ── Heatmap 2: Job Role × Work-Life Balance ───────────────
wlb_order = ["Poor","Below Average","Fair","Good","Excellent"]

pivot2 = (
    df.groupby(["job_role","work_life_balance"], observed=True)["attrition"]
      .mean().mul(100).round(1).reset_index()
      .pivot(index="job_role", columns="work_life_balance", values="attrition")
      .reindex(columns=[c for c in wlb_order if c in df["work_life_balance"].unique()])
)

fig2 = px.imshow(pivot2, text_auto=".1f",
    color_continuous_scale=["#22c55e","#fcd34d","#ef4444"],
    aspect="auto",
    title="Heatmap 2: Attrition Rate (%) — Job Role × Work-Life Balance",
    labels=dict(color="Attrition %", x="Work-Life Balance", y="Job Role"),
)
fig2.update_layout(height=380, template="plotly_white")
fig2.show()

## Final EDA Summary: Feature Impact Ranking


In [ ]:
summary_rows = []

# Categorical features: use max attrition rate - min attrition rate as impact measure
categorical_features = [
    ("work_life_balance",   ["Poor","Below Average","Fair","Good","Excellent"]),
    ("job_satisfaction",    ["Very Low","Low","Medium","High","Very High"]),
    ("employee_recognition",["Very Low","Low","Medium","High","Very High"]),
    ("company_reputation",  ["Very Poor","Poor","Fair","Good","Excellent"]),
    ("job_role",            None),
    ("job_level",           ["Entry","Mid","Senior"]),
    ("marital_status",      None),
    ("gender",              None),
    ("education_level",     None),
    ("company_size",        ["Small","Medium","Large"]),
    ("remote_work",         None),
]

for col, order in categorical_features:
    if col not in df.columns: continue
    agg = attrition_rate(df, col, order)
    rate_max  = agg["rate_pct"].max()
    rate_min  = agg["rate_pct"].min()
    spread    = rate_max - rate_min
    chi2, p, dof, sig = chi_square_test(df, col)
    highest_risk = agg.iloc[0][col] if order is None else agg[agg["rate_pct"]==rate_max].iloc[0][col]
    summary_rows.append({
        "Feature"      : col,
        "Type"         : "Categorical",
        "Impact Score" : round(spread, 1),
        "Metric"       : f"Rate spread: {rate_min:.1f}% → {rate_max:.1f}%",
        "Highest Risk Group": str(highest_risk),
        "Statistical Sig": sig,
    })

# Numeric features: use point-biserial correlation as impact
for row in results:
    col = row["feature"]
    if col not in df.columns: continue
    summary_rows.append({
        "Feature"      : col,
        "Type"         : "Numeric",
        "Impact Score" : round(abs(row["point_biserial"]) * 100, 1),
        "Metric"       : f"Point-biserial: {row['point_biserial']:+.4f}",
        "Highest Risk Group": "Higher values → more likely to leave" if row["point_biserial"] > 0 else "Lower values → more likely to leave",
        "Statistical Sig": row["significant"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Impact Score", ascending=False)
summary_df.index = range(1, len(summary_df)+1)
summary_df.index.name = "Rank"

print("🏆 FEATURE IMPACT RANKING — Strongest predictors of attrition:")
display(summary_df)

print("\n📌 HR RECOMMENDATIONS (based on top-ranked controllable factors):")
recommendations = [
    "1. Work-Life Balance: Highest controllable impact — implement flexible scheduling",
    "2. Job Satisfaction: Regular pulse surveys + manager 1-on-1 programs",
    "3. Employee Recognition: Low-cost, high-impact — formal recognition programs",
    "4. Promotion Pathways: Define clear criteria; 0-promotion employees leave most",
    "5. Compensation Review: Benchmark salaries by role, especially for Entry-level",
    "6. Remote Work: Evaluate WFH policies — can reduce commute-related attrition",
]
for r in recommendations:
    print(f"   {r}")

🏆 FEATURE IMPACT RANKING — Strongest predictors of attrition:


,Feature,Type,Impact Score,Metric,Highest Risk Group,Statistical Sig
Rank,,,,,,
1,job_level,Categorical,43.00,Rate spread: 20.3% → 63.3%,Entry,*** (p<0.001)
2,job_level_num,Numeric,31.50,Point-biserial: -0.3153,Lower values → more likely to leave,✅
3,marital_status,Categorical,30.70,Rate spread: 36.0% → 66.8%,Single,*** (p<0.001)
4,remote_work,Categorical,28.10,Rate spread: 24.7% → 52.8%,0.0,*** (p<0.001)
5,education_level,Categorical,24.50,Rate spread: 24.4% → 49.0%,Bachelor's Degree,*** (p<0.001)
6,work_life_balance,Categorical,24.50,Rate spread: 35.6% → 60.2%,Poor,*** (p<0.001)
7,remote_work,Numeric,22.10,Point-biserial: -0.2212,Lower values → more likely to leave,✅
8,work_life_balance_num,Numeric,17.50,Point-biserial: -0.1747,Lower values → more likely to leave,✅
9,company_reputation,Categorical,13.00,Rate spread: 43.0% → 56.0%,Poor,*** (p<0.001)



📌 HR RECOMMENDATIONS (based on top-ranked controllable factors):
   1. Work-Life Balance: Highest controllable impact — implement flexible scheduling
   2. Job Satisfaction: Regular pulse surveys + manager 1-on-1 programs
   3. Employee Recognition: Low-cost, high-impact — formal recognition programs
   4. Promotion Pathways: Define clear criteria; 0-promotion employees leave most
   5. Compensation Review: Benchmark salaries by role, especially for Entry-level
   6. Remote Work: Evaluate WFH policies — can reduce commute-related attrition


---
# 🟥 EXPORT
---

## Save Clean Dataset



In [ ]:
df.to_csv("hr_attrition_clean.csv", index=False)

print("✅ Exported: hr_attrition_clean.csv")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"   Size    : ~{df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumns in exported file:")
for i, c in enumerate(df.columns, 1):
    dtype = str(df[c].dtype)
    print(f"  {i:2d}. {c:<35} ({dtype})")

# Download in Colab
try:
    from google.colab import files
    files.download("hr_attrition_clean.csv")
    print("\n📥 Download triggered — check your browser downloads folder")
except ImportError:
    print("\n(Not in Colab — file saved to current working directory)")

✅ Exported: hr_attrition_clean.csv
   Rows    : 74,498
   Columns : 38
   Size    : ~64.2 MB

Columns in exported file:
   1. employee_id                         (int64)
   2. age                                 (int64)
   3. gender                              (object)
   4. years_at_company                    (int64)
   5. job_role                            (object)
   6. monthly_income                      (float64)
   7. work_life_balance                   (object)
   8. job_satisfaction                    (object)
   9. performance_rating                  (object)
  10. number_of_promotions                (int64)
  11. overtime                            (object)
  12. distance_from_home                  (int64)
  13. education_level                     (object)
  14. marital_status                      (object)
  15. number_of_dependents                (int64)
  16. job_level                           (object)
  17. company_size                        (object)
  18. company_tenu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Download triggered — check your browser downloads folder
